# Formatos de Datos: XML y JSON
## Acceso a APIs científicas con Python

**Temas:**
- ¿Qué son XML y JSON y para qué sirven?
- Casos de uso en física y matemática
- ¿Qué es una API y un endpoint?
- La librería `requests`
- Consultar la API de la NASA y explorar los datos

**Requisitos:**
```bash
pip install requests
```

---
## Parte 1 — ¿Por qué necesitamos formatos de datos?

Imagina que haces una simulación en Python y quieres:
- Guardar los resultados para analizarlos después
- Enviarlos a un colega que usa MATLAB
- Compartirlos con un servidor web

Si guardas los datos en un formato inventado por ti, nadie más podrá leerlos.

Los **formatos estándar** como **XML** y **JSON** resuelven este problema: son texto plano, legible por humanos, y existe software para leerlos en prácticamente cualquier lenguaje de programación.

| Situación | Sin formato estándar | Con formato estándar |
|---|---|---|
| Compartir resultados | Cada uno inventa su propio formato | Todos leen el mismo archivo |
| Comunicar con un servidor | Imposible sin acordar un protocolo | JSON/XML es el estándar de internet |
| Archivar datos de experimento | Formato propietario, puede quedar obsoleto | XML/JSON legible en cualquier época |

---
## Parte 2 — XML (eXtensible Markup Language)

XML es un formato de texto que organiza datos usando **etiquetas** (tags), similares a las de HTML.  
Cada etiqueta tiene una apertura `<tag>` y un cierre `</tag>`.

### Estructura básica

```xml
<?xml version="1.0" encoding="UTF-8"?>
<experimento>
    <nombre>Caída libre</nombre>
    <fecha>2025-04-12</fecha>
    <mediciones>
        <punto tiempo="0.0" posicion="0.00"/>
        <punto tiempo="0.5" posicion="1.23"/>
        <punto tiempo="1.0" posicion="4.90"/>
    </mediciones>
</experimento>
```

**Conceptos clave:**
- **Elemento raíz**: todo el XML tiene un solo elemento padre (`<experimento>`)
- **Etiquetas**: delimitan el contenido (`<nombre>`, `<fecha>`)
- **Atributos**: información extra dentro de la etiqueta (`tiempo="0.5"`)
- **Anidado**: los elementos pueden contener otros elementos

### Casos de uso en física y matemática

- Formato **MathML**: representar fórmulas matemáticas en documentos
- Formato **CML** (Chemical Markup Language): moléculas y reacciones
- Archivos **SVG**: gráficas vectoriales generadas por simulaciones
- Configuración de simulaciones en software científico (GEANT4, OpenFOAM)
- Intercambio de datos entre laboratorios (protocolo SDMX en estadística)

### Leer XML con Python — `xml.etree.ElementTree`

Python incluye este módulo en su librería estándar (no hay que instalar nada).

In [ ]:
import xml.etree.ElementTree as ET

# Datos de un experimento de caída libre (como si vinieran de un archivo .xml)
xml_texto = """<?xml version="1.0" encoding="UTF-8"?>
<experimento nombre="Caida libre" lugar="Laboratorio F12">
    <descripcion>Medicion de posicion de un objeto en caida libre</descripcion>
    <constantes>
        <g unidad="m/s2">9.8</g>
        <h0 unidad="m">20.0</h0>
    </constantes>
    <mediciones>
        <punto t="0.0" y="20.00" vy="0.00"/>
        <punto t="0.5" y="18.78" vy="4.90"/>
        <punto t="1.0" y="15.10" vy="9.80"/>
        <punto t="1.5" y="8.98" vy="14.70"/>
        <punto t="2.0" y="0.40" vy="19.60"/>
    </mediciones>
</experimento>
"""

# Parsear el texto XML
raiz = ET.fromstring(xml_texto)

# Leer atributos del elemento raíz
print("Experimento:", raiz.get("nombre"))
print("Lugar      :", raiz.get("lugar"))

# Leer un elemento hijo
descripcion = raiz.find("descripcion")
print("Descripción:", descripcion.text)

# Leer las constantes
constantes = raiz.find("constantes")
g  = float(constantes.find("g").text)
h0 = float(constantes.find("h0").text)
print(f"\ng = {g} m/s²   h0 = {h0} m")

In [ ]:
# Iterar sobre las mediciones
print("\nDatos experimentales:")
print(f"{'t (s)':>8}  {'y (m)':>8}  {'vy (m/s)':>10}")
print("-" * 32)

mediciones = raiz.find("mediciones")
for punto in mediciones:
    t  = float(punto.get("t"))
    y  = float(punto.get("y"))
    vy = float(punto.get("vy"))
    print(f"{t:>8.1f}  {y:>8.2f}  {vy:>10.2f}")

# También podemos reconstruir los datos como listas de Python
tiempos   = [float(p.get("t"))  for p in mediciones]
posiciones = [float(p.get("y")) for p in mediciones]

print("\nListas reconstruidas:")
print("t :", tiempos)
print("y :", posiciones)

---
## Parte 3 — JSON (JavaScript Object Notation)

JSON es el formato más usado hoy en día para intercambiar datos en internet.  
Su sintaxis es muy similar a los **diccionarios y listas de Python**.

### Estructura básica

```json
{
    "experimento": "Caída libre",
    "fecha": "2025-04-12",
    "constantes": {
        "g": 9.8,
        "h0": 20.0
    },
    "mediciones": [
        {"t": 0.0, "y": 20.00, "vy": 0.00},
        {"t": 0.5, "y": 18.78, "vy": 4.90},
        {"t": 1.0, "y": 15.10, "vy": 9.80}
    ]
}
```

**Tipos de datos en JSON:**

| Tipo JSON | Equivalente Python | Ejemplo |
|---|---|---|
| objeto `{}` | `dict` | `{"clave": "valor"}` |
| arreglo `[]` | `list` | `[1, 2, 3]` |
| cadena `""` | `str` | `"texto"` |
| número | `int` / `float` | `42`, `3.14` |
| booleano | `bool` | `true` / `false` |
| nulo | `None` | `null` |

### Casos de uso en física y matemática

- Resultados de simulaciones (LIGO, LHC publican datos en JSON)
- APIs de datos meteorológicos, sísmicos, astronómicos
- Configuración de experimentos y pipelines de análisis
- Bases de datos de constantes físicas (NIST publica en JSON)
- Intercambio entre Jupyter notebooks y dashboards interactivos (Plotly, D3.js)

### XML vs JSON — ¿Cuándo usar cada uno?

| Característica | XML | JSON |
|---|---|---|
| Legibilidad | Más verboso | Más compacto |
| Soporte en Python | `xml.etree.ElementTree` | `json` (built-in) |
| Atributos en metadatos | Sí (nativo) | No (se simulan con claves) |
| APIs web modernas | Poco común | Estándar actual |
| Documentos científicos | MathML, CML, SVG | Datos de sensores, APIs |
| Comentarios | Sí (`<!-- -->`) | No |

**Regla práctica:** si vas a consumir datos de una API en internet → JSON.  
Si trabajas con documentos estructurados (fórmulas, gráficas vectoriales) → XML.

In [ ]:
import json

# El módulo json convierte entre texto JSON y objetos Python
# json.loads()  → texto JSON  → dict/list de Python
# json.dumps()  → dict/list   → texto JSON

experimento = {
    "nombre": "Péndulo simple",
    "fecha": "2025-04-12",
    "longitud_m": 1.0,
    "mediciones": [
        {"t": 0.00, "theta": 15.0},
        {"t": 0.25, "theta":  8.3},
        {"t": 0.50, "theta": -0.1},
        {"t": 0.75, "theta": -8.4},
        {"t": 1.00, "theta":-14.9},
    ]
}

# Convertir el dict a texto JSON (indent para hacerlo legible)
texto_json = json.dumps(experimento, indent=2, ensure_ascii=False)
print("── Texto JSON ──")
print(texto_json)

In [ ]:
# Leer el JSON de vuelta a un diccionario de Python
datos = json.loads(texto_json)

print("Tipo del resultado:", type(datos))
print("Nombre            :", datos["nombre"])
print("Longitud (m)      :", datos["longitud_m"])

print("\nMediciones del péndulo:")
for m in datos["mediciones"]:
    print(f"  t = {m['t']:.2f} s   θ = {m['theta']:+.1f}°")

---
## Parte 4 — ¿Qué es una API y un endpoint?

### API (Application Programming Interface)

Una **API** es una interfaz que permite que dos programas se comuniquen entre sí.  
Piénsalo como el menú de un restaurante: tú no entras a la cocina, simplemente pides un platillo (haces una solicitud) y el restaurante te lo entrega (responde).

En el caso de las APIs web:
- **Tu programa** = el cliente (quien pide)
- **El servidor** = el restaurante (quien responde)
- **El protocolo** = HTTP (el idioma que ambos hablan)

### Endpoint

Un **endpoint** es la dirección específica (URL) que corresponde a un recurso o acción particular dentro de una API.

```
https://api.nasa.gov / planetary / apod ? api_key=DEMO_KEY
───────────────────── ─────────── ──── ──────────────────
   servidor base          ruta    recurso   parámetros
```

Una misma API puede tener muchos endpoints, cada uno para datos distintos:
- `.../planetary/apod` → Imagen astronómica del día
- `.../neo/rest/v1/feed` → Asteroides cerca de la Tierra
- `.../mars-photos/...` → Fotos del rover en Marte

### La librería `requests`

`requests` es la librería estándar de Python para hacer solicitudes HTTP.  
El flujo básico siempre es el mismo:

```python
import requests

respuesta = requests.get("https://url-del-endpoint")  # 1. Solicitar
datos = respuesta.json()                               # 2. Convertir a dict
# ... usar los datos                                   # 3. Explorar
```

Los códigos de estado HTTP más importantes:
| Código | Significado |
|---|---|
| `200` | OK — la solicitud fue exitosa |
| `400` | Bad Request — parámetros incorrectos |
| `401` | Unauthorized — falta o es inválida la API key |
| `404` | Not Found — el endpoint no existe |
| `429` | Too Many Requests — se excedió el límite de solicitudes |

---
## Códigos de estado HTTP

Toda respuesta HTTP incluye un **código de estado numérico** de 3 dígitos que indica si la solicitud fue exitosa o qué tipo de error ocurrió. El primer dígito define la categoría:

| Rango | Categoría | Significado general |
|:---:|---|---|
| 2xx | ✅ Éxito | La solicitud se procesó correctamente |
| 3xx | ↪ Redirección | El recurso se movió a otra URL |
| 4xx | ❌ Error del cliente | El problema está en tu solicitud |
| 5xx | 🔥 Error del servidor | El problema está en el servidor |

### Códigos más comunes en APIs científicas

| Código | Nombre | Cuándo ocurre | Qué hacer |
|:---:|---|---|---|
| **200** | OK | Todo salió bien | Proceder a leer el JSON |
| **400** | Bad Request | Parámetro mal escrito o faltante | Revisar los parámetros de la URL |
| **401** | Unauthorized | No enviaste API key | Agregar `api_key` a la URL |
| **403** | Forbidden | API key inválida o revocada | Verificar o regenerar la key |
| **404** | Not Found | El endpoint no existe | Revisar la URL |
| **429** | Too Many Requests | Excediste el límite de solicitudes | Esperar antes de reintentar |
| **500** | Internal Server Error | Bug en el servidor | Reportar al proveedor de la API |
| **503** | Service Unavailable | Servidor caído o en mantenimiento | Reintentar más tarde |

> **Regla práctica:** códigos **2xx** → éxito. Cualquier otro → hay un problema que debes manejar antes de llamar `.json()`.

In [ ]:
import requests

# ── Ejemplo 1: verificación básica con if/else ──────────────────────────────
url = "https://api.nasa.gov/planetary/apod?api_key=DEMO_KEY"
respuesta = requests.get(url, timeout=15)

if respuesta.status_code == 200:
    datos = respuesta.json()
    print("✓ 200 OK — datos recibidos:", list(datos.keys()))
else:
    print(f"✗ Error {respuesta.status_code}")

In [ ]:
# ── Ejemplo 2: usar respuesta.ok (True si 200-399) ──────────────────────────
respuesta = requests.get(url, timeout=15)

if respuesta.ok:
    datos = respuesta.json()
    print("✓ Solicitud exitosa")
else:
    print(f"✗ Falló con código {respuesta.status_code}")

In [ ]:
# ── Ejemplo 3: manejar cada código con mensajes útiles ──────────────────────
respuesta = requests.get(url, timeout=10)

codigo = respuesta.status_code

if codigo == 200:
    datos = respuesta.json()
    print("✓ OK — procesando datos...")
elif codigo == 400:
    print("✗ 400 Bad Request — revisa los parámetros de la URL")
elif codigo == 401:
    print("✗ 401 Unauthorized — falta la API key en la URL")
elif codigo == 403:
    print("✗ 403 Forbidden — API key inválida o revocada")
elif codigo == 404:
    print("✗ 404 Not Found — el endpoint no existe, revisa la URL")
elif codigo == 429:
    print("✗ 429 Too Many Requests — espera unos minutos y vuelve a intentar")
elif codigo >= 500:
    print(f"✗ {codigo} Error del servidor — el problema está en la API, intenta más tarde")
else:
    print(f"✗ Código inesperado: {codigo}")

In [ ]:
# ── Ejemplo 4: raise_for_status() — lanza una excepción automáticamente ─────
# En lugar de verificar manualmente, requests puede lanzar un error
# si el código no es 2xx. Útil cuando quieres detener el programa de inmediato.

try:
    respuesta = requests.get(url, timeout=10)
    respuesta.raise_for_status()   # lanza HTTPError si status >= 400
    datos = respuesta.json()
    print("✓ Datos recibidos:", datos.get("title", ""))

except requests.exceptions.HTTPError as e:
    print(f"✗ Error HTTP: {e}")
except requests.exceptions.ConnectionError:
    print("✗ Sin conexión a internet o el servidor no responde")
except requests.exceptions.Timeout:
    print("✗ La solicitud tardó demasiado (timeout)")
except requests.exceptions.RequestException as e:
    # Captura cualquier otro error de requests
    print(f"✗ Error inesperado: {e}")

In [ ]:
import requests   # pip install requests
import json

print("Librería requests lista ✓")

---
## Ejemplo 1 — Posición de la Estación Espacial Internacional (ISS)

Empecemos con el ejemplo más simple: un endpoint público sin necesidad de API key.

**API:** Open Notify — `http://api.open-notify.org/iss-now.json`

Este endpoint devuelve en tiempo real la posición de la ISS (latitud y longitud).

In [ ]:
# Hacer la solicitud al endpoint
url_iss = "http://api.open-notify.org/iss-now.json"
respuesta = requests.get(url_iss)

# Verificar que la solicitud fue exitosa
print("Código de estado HTTP:", respuesta.status_code)

# Convertir la respuesta JSON a un diccionario de Python
datos_iss = respuesta.json()

print("\n── Respuesta completa ──")
print(json.dumps(datos_iss, indent=2))


In [ ]:
# Explorar el diccionario
print("Tipo de datos_iss:", type(datos_iss))
print("Claves del diccionario:", list(datos_iss.keys()))

# Extraer la posición
posicion = datos_iss["iss_position"]
lat = float(posicion["latitude"])
lon = float(posicion["longitude"])

import time
timestamp = datos_iss["timestamp"]
fecha_hora = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime(timestamp))

print(f"\nPosición actual de la ISS:")
print(f"  Latitud  : {lat:+.4f}°")
print(f"  Longitud : {lon:+.4f}°")
print(f"  Hora UTC : {fecha_hora}")
print(f"  Velocidad orbital aprox.: 7.66 km/s  (~27,600 km/h)")

---
## Variables de entorno — cómo guardar credenciales de forma segura

### ¿Qué es una variable de entorno?

Una **variable de entorno** es un par `NOMBRE=valor` que el sistema operativo
pone a disposición de todos los programas que se ejecutan en él.
Son el lugar estándar para guardar información sensible como contraseñas y API keys,
porque **nunca forman parte del código fuente** y por lo tanto no se suben a git.

```
Sistema Operativo
├── Variables de entorno
│   ├── NASA_API_KEY = "tu_key_aqui"   ← solo existe en tu máquina
│   ├── HOME = "/home/usuario"
│   └── PATH = "/usr/bin:/usr/local/bin"
└── Tu programa Python lee estas variables con os.getenv()
```

### ¿Por qué NO poner la API key directo en el código?

```python
# ❌ MAL — la key queda expuesta en el archivo, en git, en GitHub...
API_KEY = "4VAQSAtYt6DrjgGz..."

# ✅ BIEN — el código no contiene la key, cada quien pone la suya en su máquina
API_KEY = os.getenv("NASA_API_KEY", "DEMO_KEY")
```

Si subes el código a GitHub con la key hardcodeada, bots automáticos la escanean
y pueden abusar de tu cuenta en minutos.

### Cómo configurar la variable de entorno

**Linux / macOS — sesión actual (temporal):**
```bash
export NASA_API_KEY="tu_api_key_aqui"
```

**Linux / macOS — permanente (agregar al archivo `~/.bashrc` o `~/.zshrc`):**
```bash
echo 'export NASA_API_KEY="tu_api_key_aqui"' >> ~/.bashrc
source ~/.bashrc
```

**Windows (PowerShell):**
```powershell
$env:NASA_API_KEY = "tu_api_key_aqui"
```

**Desde Python directamente (solo para pruebas, no en producción):**
```python
import os
os.environ["NASA_API_KEY"] = "tu_api_key_aqui"
```

### Archivo `.env` con `python-dotenv` (opción recomendada para proyectos)

Crea un archivo llamado `.env` en tu proyecto (y agrégalo a `.gitignore`):
```
# .env  ← este archivo NUNCA se sube a git
NASA_API_KEY=tu_api_key_aqui
```
Luego en Python:
```bash
pip install python-dotenv
```
```python
from dotenv import load_dotenv
load_dotenv()                         # carga las variables del archivo .env
API_KEY = os.getenv("NASA_API_KEY")
```

### Práctica: archivo `.env` con `python-dotenv`

El método con variables del sistema funciona bien en servidores, pero en proyectos locales
(como este notebook) es más cómodo usar un archivo `.env`:

1. Crea el archivo `.env` en la misma carpeta del notebook (la celda siguiente lo hace por ti)
2. Agrega `.env` a tu `.gitignore` para que **nunca se suba a git**
3. Llama a `load_dotenv()` al inicio del notebook, antes de cualquier `os.getenv()`

```
# .gitignore
.env
*.env
```

> **Jerarquía de prioridad** (de mayor a menor):  
> `Variables del sistema` > `archivo .env` > valor por defecto en `os.getenv()`
>
> Esto significa que si ya configuraste `NASA_API_KEY` en tu sistema,
> `load_dotenv()` **no la sobreescribe** — tu configuración local siempre gana.


In [ ]:
# Instalar python-dotenv (solo la primera vez; --quiet suprime la salida larga)
%pip install python-dotenv --quiet

from pathlib import Path
from dotenv import load_dotenv, dotenv_values

# --- Paso 1: crear .env de ejemplo si no existe ---
env_file = Path(".env")

if not env_file.exists():
    env_file.write_text(
        "# .env — NUNCA subir a git (agrégalo a .gitignore)\n"
        "# Descomenta la línea siguiente y reemplaza con tu API key real:\n"
        "# NASA_API_KEY=tu_api_key_aqui\n"
    )
    print("✓ Archivo .env creado. Edítalo con tu NASA API key.")
else:
    print(f"✓ .env encontrado en: {env_file.resolve()}")

# --- Paso 2: cargar variables desde .env hacia os.environ ---
# override=False (por defecto): si la variable ya existe en el sistema, NO la sobreescribe.
# Esto garantiza que la configuración del sistema siempre tiene prioridad sobre .env.
cargadas = load_dotenv(env_file, override=False)
print(f"Variables cargadas desde .env: {'sí' if cargadas else 'archivo vacío o sin variables activas'}")

# --- Paso 3: inspeccionar qué hay en .env (sin exponer valores reales) ---
vars_env = dotenv_values(env_file)
if vars_env:
    print("\nVariables definidas en .env:")
    for clave, valor in vars_env.items():
        if valor:
            print(f"  {clave} = {'*' * min(len(valor), 8)}  (valor oculto por seguridad)")
        else:
            print(f"  {clave} = (vacía o comentada)")
else:
    print("\n  (ninguna variable activa en .env — descomenta NASA_API_KEY para activarla)")


In [ ]:
import os, requests, time
from dotenv import load_dotenv

# load_dotenv() lee el archivo .env del directorio actual y carga las variables.
# Si la variable ya existe en os.environ (sistema), NO la sobreescribe.
# Llamarlo aquí es seguro aunque ya se llamó antes: es idempotente.
load_dotenv()

# os.getenv devuelve "" si la variable existe pero está vacía,
# por eso usamos .strip() or "DEMO_KEY" en lugar del segundo argumento solo.
API_KEY = os.getenv("NASA_API_KEY", "").strip() or "DEMO_KEY"

if API_KEY == "DEMO_KEY":
    print("⚠  NASA_API_KEY no configurada → usando DEMO_KEY (30 req/hora)")
else:
    print(f"✓  API key cargada: {API_KEY[:4]}{'*' * (len(API_KEY) - 4)}")


---
## Ejemplo 2 — NASA: Imagen Astronómica del Día (APOD)

**API:** NASA Open APIs — `https://api.nasa.gov`  
**Endpoint:** `/planetary/apod`  
**API Key:** usaremos `DEMO_KEY` (funciona con límite de 30 solicitudes/hora)

> Para proyectos más serios puedes registrarte gratis en https://api.nasa.gov  
> y obtener tu propia key con 1000 solicitudes/hora.

In [ ]:
# Algunos endpoints de NASA devuelven 503 de forma intermitente.
# Este código reintenta hasta 3 veces antes de rendirse.

MAX_INTENTOS = 3
datos_apod = None

for intento in range(1, MAX_INTENTOS + 1):
    print(f"Intento {intento}/{MAX_INTENTOS}...", end=" ")
    respuesta = requests.get(
        f"https://api.nasa.gov/planetary/apod?api_key={API_KEY}",
        timeout=10
    )
    print(f"HTTP {respuesta.status_code}")

    if respuesta.status_code == 200:
        datos_apod = respuesta.json()
        break
    elif respuesta.status_code == 503:
        print("   Servidor no disponible, esperando 5 segundos...")
        if intento < MAX_INTENTOS:
            time.sleep(5)
    elif respuesta.status_code == 429:
        print("   Límite de solicitudes excedido, espera unos minutos.")
        break
    elif respuesta.status_code in (401, 403):
        print("   API key inválida o revocada.")
        break
    else:
        print(f"   Error inesperado: {respuesta.text[:100]}")
        break

if datos_apod is None:
    raise SystemExit("No se pudo obtener respuesta del servidor. Intenta más tarde.")

print("\n── Claves del JSON ──")
for clave, valor in datos_apod.items():
    val_str = str(valor)
    if len(val_str) > 80:
        val_str = val_str[:77] + "..."
    print(f"  {clave:20} : {val_str}")

In [ ]:
# Explorar los datos con más detalle
print("=" * 60)
print("IMAGEN ASTRONÓMICA DEL DÍA")
print("=" * 60)
print(f"Título : {datos_apod['title']}")
print(f"Fecha  : {datos_apod['date']}")
print(f"Tipo   : {datos_apod['media_type']}")

if "copyright" in datos_apod:
    print(f"Autor  : {datos_apod['copyright'].strip()}")

print(f"\nDescripción:")
# Imprimir el texto en párrafos de ~80 caracteres
explicacion = datos_apod["explanation"]
palabras = explicacion.split()
linea = ""
for palabra in palabras:
    if len(linea) + len(palabra) + 1 > 80:
        print(" ", linea)
        linea = palabra
    else:
        linea = linea + " " + palabra if linea else palabra
if linea:
    print(" ", linea)

print(f"\nURL de la imagen:")
print(" ", datos_apod.get("url", "N/A"))

In [ ]:
from IPython.display import Image, IFrame, display

media_type = datos_apod.get("media_type", "image")

if media_type == "image":
    print(f"Mostrando: {datos_apod['title']}")
    display(Image(url=datos_apod["url"], width=700))
elif media_type == "video":
    # APOD a veces es un video de YouTube — lo mostramos en un iframe
    print(f"El APOD de hoy es un video: {datos_apod['title']}")
    print(f"URL: {datos_apod['url']}")
    display(IFrame(src=datos_apod["url"], width=700, height=400))
else:
    print(f"Tipo de media no reconocido: {media_type}")
    print("URL:", datos_apod.get("url", "N/A"))

---
## Ejemplo 3 — NASA: Asteroides cerca de la Tierra (NEO)

**Endpoint:** `/neo/rest/v1/feed`  
**Parámetros:** `start_date`, `end_date`, `api_key`

Este endpoint devuelve todos los asteroides (Near-Earth Objects) que pasaron cerca de la Tierra en un rango de fechas. Es un buen ejemplo de JSON **anidado** con listas dentro de diccionarios.

In [ ]:
url_neo = (
    "https://api.nasa.gov/neo/rest/v1/feed"
    "?start_date=2025-01-01"
    "&end_date=2025-01-03"
    f"&api_key={API_KEY}"
)

In [ ]:

respuesta = requests.get(url_neo)
print("Código HTTP:", respuesta.status_code)

if not respuesta.ok:
    print(f"Error del servidor: {respuesta.status_code}")
    print("Respuesta recibida:", respuesta.text[:200] or "(vacía)")
    raise SystemExit("Deteniéndose por error HTTP.")

datos_neo = respuesta.json()

In [ ]:
datos_neo

In [ ]:
print("\nClaves en la respuesta:", list(datos_neo.keys()))
print("Total de asteroides en el período:", datos_neo["element_count"])
fechas = list(datos_neo["near_earth_objects"].keys())
print("Fechas disponibles:", fechas)

In [ ]:
datos_neo["near_earth_objects"]['2025-01-03'][3]

In [ ]:
# Explorar la estructura de UN asteroide para entender el JSON
primera_fecha = fechas[0]
primer_asteroide = datos_neo["near_earth_objects"][primera_fecha][0]

print(f"── Primer asteroide del {primera_fecha} ──")
print(f"Nombre : {primer_asteroide['name']}")
print(f"ID     : {primer_asteroide['id']}")
print(f"Mag. absoluta (H): {primer_asteroide['absolute_magnitude_h']}")
print(f"¿Potencialmente peligroso?: {primer_asteroide['is_potentially_hazardous_asteroid']}")

# El diámetro estimado es un sub-diccionario
diametro = primer_asteroide["estimated_diameter"]["meters"]
print(f"Diámetro estimado: {diametro['estimated_diameter_min']:.1f} – {diametro['estimated_diameter_max']:.1f} m")

# Los datos de aproximación también son una lista
acercamiento = primer_asteroide["close_approach_data"][0]
print(f"Fecha acercamiento : {acercamiento['close_approach_date']}")
print(f"Distancia (km)     : {float(acercamiento['miss_distance']['kilometers']):,.0f} km")
print(f"Velocidad (km/h)   : {float(acercamiento['relative_velocity']['kilometers_per_hour']):,.0f} km/h")

In [ ]:
# ── Exploración con ciclos ──────────────────────────────────────────────────
# Recorrer TODOS los asteroides de TODAS las fechas y hacer un resumen

print(f"{'Nombre':<30} {'Diám. máx (m)':>14} {'Dist. (km)':>16} {'Peligroso':>10}")
print("-" * 75)

total    = 0
peligrosos = 0

for fecha in sorted(fechas):
    asteroides_del_dia = datos_neo["near_earth_objects"][fecha]

    for ast in asteroides_del_dia:
        total += 1

        nombre   = ast["name"][:28]
        diam_max = ast["estimated_diameter"]["meters"]["estimated_diameter_max"]
        dist_km  = float(ast["close_approach_data"][0]["miss_distance"]["kilometers"])
        peligro  = ast["is_potentially_hazardous_asteroid"]

        if peligro:
            peligrosos += 1

        marca = "⚠ SÍ" if peligro else "no"
        print(f"{nombre:<30} {diam_max:>14.1f} {dist_km:>16,.0f} {marca:>10}")

print("-" * 75)
print(f"Total: {total} asteroides   |   Potencialmente peligrosos: {peligrosos}")

In [ ]:
# ── Estadísticas básicas con ciclos ────────────────────────────────────────
diametros = []
distancias = []
velocidades = []

for fecha in fechas:
    for ast in datos_neo["near_earth_objects"][fecha]:
        diametros.append(
            ast["estimated_diameter"]["meters"]["estimated_diameter_max"]
        )
        distancias.append(
            float(ast["close_approach_data"][0]["miss_distance"]["kilometers"])
        )
        velocidades.append(
            float(ast["close_approach_data"][0]["relative_velocity"]["kilometers_per_hour"])
        )

# Calcular estadísticas con ciclos (sin librerías externas)
def promedio(lista):
    return sum(lista) / len(lista)

print("── Estadísticas del período ──")
print(f"Asteroides analizados : {len(diametros)}")
print(f"\nDiámetro máximo estimado:")
print(f"  Promedio : {promedio(diametros):>10.1f} m")
print(f"  Máximo   : {max(diametros):>10.1f} m")
print(f"  Mínimo   : {min(diametros):>10.1f} m")
print(f"\nDistancia de acercamiento (miss distance):")
print(f"  Promedio : {promedio(distancias):>15,.0f} km")
print(f"  Más cerca: {min(distancias):>15,.0f} km")
print(f"  Más lejos: {max(distancias):>15,.0f} km")
print(f"\nVelocidad relativa:")
print(f"  Promedio : {promedio(velocidades):>12,.0f} km/h")
print(f"  Máxima   : {max(velocidades):>12,.0f} km/h")

---
## Ejemplo 4 — NASA EONET: Eventos Naturales en la Tierra

**EONET** = Earth Observatory Natural Event Tracker  
**API:** `https://eonet.gsfc.nasa.gov/api/v3`  
**Sin API key** — es completamente pública.

Registra eventos naturales en tiempo real: incendios, tormentas, volcanes, terremotos, inundaciones, etc.

### Parámetros del endpoint `/events`

| Parámetro | Tipo | Default | Descripción |
|---|---|---|---|
| `limit` | int | 10 | Máximo de eventos a retornar |
| `days` | int | 60 | Rango en días hacia atrás desde hoy |
| `status` | string | `open` | `open` = activos, `closed` = finalizados, `all` = ambos |
| `category` | string | todas | Filtrar por categoría (ver tabla abajo) |

### Categorías disponibles

| ID | Nombre |
|---|---|
| `wildfires` | Incendios forestales |
| `severeStorms` | Tormentas severas |
| `volcanoes` | Volcanes |
| `earthquakes` | Terremotos |
| `floods` | Inundaciones |
| `seaLakeIce` | Hielo marino |
| `drought` | Sequías |

### Estructura de un evento

```json
{
  "id":    "EONET_19481",
  "title": "Super Typhoon Sinlaku",
  "closed": null,
  "categories": [{ "id": "severeStorms", "title": "Severe Storms" }],
  "geometry": [
    {
      "date":           "2026-04-09T18:00:00Z",
      "magnitudeValue": 35.0,
      "magnitudeUnit":  "kts",
      "coordinates":    [151.5, 8.3]
    }
  ]
}
```

**Campos importantes:**
- `closed` — `null` si el evento sigue activo; fecha de cierre si ya terminó
- `geometry` — lista de posiciones ordenadas por tiempo; cada punto tiene coordenadas `[longitud, latitud]`
- `magnitudeValue` / `magnitudeUnit` — intensidad del evento (kts para tormentas, hectáreas para incendios, etc.)

In [ ]:
# EONET no requiere API key
url_eonet = "https://eonet.gsfc.nasa.gov/api/v3/events?limit=20&days=30&status=all"

respuesta = requests.get(url_eonet, timeout=10)
print("Código HTTP:", respuesta.status_code)

if not respuesta.ok:
    raise SystemExit(f"Error {respuesta.status_code}: {respuesta.text[:200]}")

datos_eonet = respuesta.json()

print(f"Título    : {datos_eonet['title']}")
print(f"Eventos   : {len(datos_eonet['events'])}")
print(f"\nClaves de cada evento:", list(datos_eonet['events'][0].keys()))

In [ ]:
# Explorar la estructura de un evento
primer_evento = datos_eonet['events'][0]

print("=" * 55)
print(f"ID       : {primer_evento['id']}")
print(f"Título   : {primer_evento['title']}")
print(f"Categoría: {primer_evento['categories'][0]['title']}")
estado = "Activo" if primer_evento['closed'] is None else f"Cerrado: {primer_evento['closed']}"
print(f"Estado   : {estado}")
print(f"Lecturas : {len(primer_evento['geometry'])} puntos de trayectoria")

# Ver las últimas 3 posiciones
print(f"\nÚltimas posiciones registradas:")
print(f"  {'Fecha':<25} {'Lon':>8} {'Lat':>8} {'Magnitud':>12}")
print("  " + "-" * 58)
for punto in primer_evento['geometry'][-3:]:
    lon, lat = punto['coordinates']
    mag  = punto.get('magnitudeValue')
    unit = punto.get('magnitudeUnit', '')
    mag_str = f"{mag} {unit}" if mag else "N/A"
    print(f"  {punto['date']:<25} {lon:>8.2f} {lat:>8.2f} {mag_str:>12}")

In [ ]:
# Recorrer todos los eventos y agrupar por categoría
print(f"{'Categoría':<22} {'Estado':<10} {'Título'}")
print("-" * 70)

conteo_categorias = {}

for evento in datos_eonet['events']:
    categoria = evento['categories'][0]['title']
    estado    = "Activo" if evento['closed'] is None else "Cerrado"
    titulo    = evento['title'][:40]

    conteo_categorias[categoria] = conteo_categorias.get(categoria, 0) + 1
    print(f"  {categoria:<20} {estado:<10} {titulo}")

print(f"\n── Resumen por categoría ──")
for cat, total in sorted(conteo_categorias.items(), key=lambda x: -x[1]):
    print(f"  {cat:<22}: {total} evento{'s' if total > 1 else ''}")

---
### Ejercicio — Practica con EONET

Usa los datos ya descargados en `datos_eonet['events']` para responder las siguientes preguntas con código Python.

In [ ]:
# Ejercicio 1 — ¿Cuántos eventos están activos (closed == None) y cuántos cerrados?
# Hint: recorre datos_eonet['events'] y revisa el campo 'closed' de cada evento

# TU CÓDIGO AQUÍ


In [ ]:
# Ejercicio 2 — Encuentra el evento con más puntos de trayectoria en 'geometry'
# (es el que ha sido rastreado por más tiempo)
# Hint: necesitas una variable auxiliar para guardar el máximo y el evento actual
# Hint: len(evento['geometry']) te da el número de puntos

# TU CÓDIGO AQUÍ


In [ ]:
# Ejercicio 3 — Para los eventos de categoría "Severe Storms" que tengan magnitud,
# imprime el título y la magnitud máxima registrada (último punto de geometry).
# Hint: filtra con  evento['categories'][0]['id'] == 'severeStorms'
# Hint: el último punto de geometry es  evento['geometry'][-1]
# Hint: la magnitud puede no existir — usa .get('magnitudeValue') para evitar errores

# TU CÓDIGO AQUÍ


---
## Extra — Guardar datos JSON en un archivo

Una vez que descargamos datos de una API, es buena práctica guardarlos localmente  
para no depender de la conexión a internet en cada ejecución.

In [ ]:
# Guardar los datos de asteroides en un archivo JSON local
nombre_archivo = "asteroides_nasa.json"

with open(nombre_archivo, "w", encoding="utf-8") as f:
    json.dump(datos_neo, f, indent=2, ensure_ascii=False)

print(f"Datos guardados en '{nombre_archivo}'")

# Leer el archivo de vuelta
with open(nombre_archivo, "r", encoding="utf-8") as f:
    datos_cargados = json.load(f)

print(f"Asteroides cargados desde archivo: {datos_cargados['element_count']}")
print("✓ Los datos guardados son idénticos a los descargados")

---
## Resumen

| Concepto | ¿Qué es? | Herramienta Python |
|---|---|---|
| **XML** | Formato de texto con etiquetas, usado en documentos científicos | `xml.etree.ElementTree` |
| **JSON** | Formato compacto basado en pares clave-valor, estándar en APIs | módulo `json` |
| **API** | Interfaz para que dos programas intercambien datos | — |
| **Endpoint** | URL específica de un recurso dentro de una API | — |
| **`requests`** | Librería para hacer solicitudes HTTP desde Python | `pip install requests` |

**Flujo completo de trabajo:**
```python
import requests, json

respuesta  = requests.get("https://api.ejemplo.com/datos?param=valor")
datos      = respuesta.json()          # → dict de Python
for item in datos["resultados"]:       # explorar con ciclos
    print(item["nombre"], item["valor"])
with open("resultados.json", "w") as f:
    json.dump(datos, f, indent=2)      # guardar localmente
```